<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

In [50]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber

fatal: destination path 'Lettura_bilanci' already exists and is not an empty directory.


In [51]:
import pdfplumber
import pandas as pd

def trova_tabella_bilancio(pdf_path, keywords = ["immobilizzazioni immateriali",
                                                 "crediti",
                                                 "immobilizzazioni materiali",
                                                 "immobilizzazioni finanziarie"]):
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None

"""
pdf_path_2= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path_2)

print(tabella)
"""

'\npdf_path_2= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"\ntabella = trova_tabella_bilancio(pdf_path_2)\n\nprint(tabella)\n'

In [52]:
import pandas as pd
import numpy as np

def summary_tabella(tabella):
    labels_dict = {
        "totale_attivo": {
            "include": ["Totale attivo"],
            "exclude": ["circolante"]
        },

        "patrimonio_netto": {
            "include": ["Totale patrimonio netto"],
            "exclude": []
        },

        "totale_attivo_circolante": {
            "include": ["Totale attivo circolante"],
            "exclude": []
        },

        "Totale_disponibilità_liquide": {
            "include": ["Totale disponibilità liquide", "IV - Disponibilità liquide"],
            "exclude": ["esercizio"]
        },

        "Totale_debiti": {
            "include": ["Totale debiti"],
            "exclude": ["verso", "rappresentati", "tributari"]
        },

        "Totale_rimanenze": {
            "include": ["Totale rimanenze", "I - Rimanenze"],
            "exclude": []
        },

        "Totale_crediti_verso_clienti": {
            "include": ["Totale crediti verso clienti", "II - Crediti"],
            "exclude": []
        },

        "Totale_debiti_verso_fornitori": {
            "include": ["Totale debiti verso fornitori"],
            "exclude": []
        },

        "Totale_debiti_verso_banche": {
            "include": ["Totale debiti verso banche"],
            "exclude": []
        },

        "avviamento": {
            "include": ["avviamento"],
            "exclude": []
        },

        "ricavi_vendite_e_prestazioni": {
            "include": ["ricavi delle vendite e delle prestazioni"],
            "exclude": []
        },

        "Utile_perdita_esercizio": {
            "include": ["Utile (perdita) dell'esercizio"],
            "exclude": []
        },

        "Risultato_prima_delle_imposte": {
            "include": ["Risultato prima delle imposte"],
            "exclude": []
        },

        "Totale_ammortamenti_e_svalutazioni": {
            "include": ["Totale ammortamenti e svalutazioni"],
            "exclude": []
        },

        "Totale_interessi_e_altri_oneri": {
            "include": ["Totale interessi e altri oneri finanziari"],
            "exclude": []
        },

        "immobilizzazioni_immateriali": {
            "include": ["Totale immobilizzazioni immateriali", "I - Immobilizzazioni immateriali"],
            "exclude": []
        },

        "immobilizzazioni_materiali": {
            "include": ["Totale immobilizzazioni materiali", "II - Immobilizzazioni materiali"],
            "exclude": []
        },

        "immobilizzazioni_finanziarie": {
            "include": ["Totale immobilizzazioni finanziarie", "III - Immobilizzazioni finanziarie"],
            "exclude": []
        },
    }

    if tabella.shape[1] == 3:
        col_label, cols_valori = 0, [1, 2]
        date = tabella.iloc[0, 1:3].tolist()
    elif tabella.shape[1] == 5:
        col_label, cols_valori = 1, [2, 3]
        date = tabella.iloc[0, 2:4].tolist()
    else:
        return pd.DataFrame()

    def norm(x):
        x = str(x).lower().replace("\n", " ")
        return " ".join(x.split())

    def riga_valida(valori):
        return any(pd.notna(v) and str(v).strip() != "" for v in valori)

    def pulisci_numero(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        if x in ("", "-"):
            return np.nan

        negativo = x.startswith("(") and x.endswith(")")
        x = x.replace("(", "").replace(")", "").strip()

        if "," in x:
            x = x.replace(".", "").replace(",", ".")
        else:
            x = x.replace(".", "")

        try:
            val = float(x)
            return -val if negativo else val
        except:
            return np.nan

    def match_testo(testo, include, exclude):
        testo = norm(testo)
        return include.lower() in testo and not any(e.lower() in testo for e in exclude)

    def calcola_passivita_correnti(tabella, col_label, cols_valori):
        testi = tabella.iloc[:, col_label].astype(str).apply(norm)

        # inizio/fine blocco debiti
        idx_inizio = testi[testi == "d) debiti"].index
        idx_fine = testi[testi.str.contains("totale passivo", na=False)].index

        if len(idx_inizio) == 0 or len(idx_fine) == 0:
            return [None] * len(cols_valori)

        idx_inizio = idx_inizio[0]
        idx_fine = next((i for i in idx_fine if i > idx_inizio), None)

        if idx_fine is None:
            return [None] * len(cols_valori)

        sotto = tabella.loc[idx_inizio:idx_fine, [col_label] + cols_valori].copy()
        sotto["label_norm"] = sotto.iloc[:, 0].astype(str).apply(norm)

        alias_correnti = [
            "esigibili entro l'esercizio successivo",
            "esigibili entro esercizio successivo",
            "entro l'esercizio successivo",
            "entro esercizio successivo"
        ]

        mask = sotto["label_norm"].apply(lambda x: any(a in x for a in alias_correnti))
        righe_correnti = sotto.loc[mask].copy()



        if righe_correnti.empty:
            return [None] * len(cols_valori)

        for col in cols_valori:
            righe_correnti[col] = righe_correnti[col].apply(pulisci_numero)

        somme = righe_correnti[cols_valori].sum(axis=0, skipna=True).tolist()
        return [None] * len(cols_valori) if all(pd.isna(v) for v in somme) else somme

    righe = {}
    col_testi = tabella.iloc[:, col_label].astype(str)

    for nome_output, regola in labels_dict.items():
        trovato = False

        for include in regola["include"]:
            mask = col_testi.apply(lambda x: match_testo(x, include, regola["exclude"]))
            if not mask.any():
                continue

            for _, row in tabella.loc[mask, cols_valori].iterrows():
                valori = row.tolist()
                if riga_valida(valori):
                    righe[nome_output] = [pulisci_numero(v) for v in valori]
                    trovato = True
                    break

            if trovato:
                break

        if not trovato:
            righe[nome_output] = [None] * len(cols_valori)

    righe["passivita_correnti"] = calcola_passivita_correnti(tabella, col_label, cols_valori)

    return pd.DataFrame.from_dict(righe, orient="index", columns=date)


In [53]:
import glob
import os

pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = trova_tabella_bilancio(pdf_path)

    if tabella is not None:
        summary[nome] = summary_tabella(tabella)

# stampa leggibile
for nome, df in summary.items():
    print("\n" + "=" * 80)
    print(f"BILANCIO: {nome}")
    print("=" * 80)
    print(df)


BILANCIO: Conserve Italia - Civilistico e Consolidato - 2024
                                    30 giugno 2025  30 giugno 2024
totale_attivo                          932834011.0     879915161.0
patrimonio_netto                       305929558.0     300611685.0
totale_attivo_circolante               439099320.0     416207934.0
Totale_disponibilità_liquide           127609819.0     106847278.0
Totale_debiti                          591475140.0     545013151.0
Totale_rimanenze                       205612371.0     201996264.0
Totale_crediti_verso_clienti                   NaN             NaN
Totale_debiti_verso_fornitori                  NaN             NaN
Totale_debiti_verso_banche             191590518.0     168282068.0
avviamento                                     NaN             NaN
ricavi_vendite_e_prestazioni           709159879.0     733550069.0
Utile_perdita_esercizio                  4787472.0       9540622.0
Risultato_prima_delle_imposte            4806517.0       9561256.0


In [55]:
print(summary["Volpato Industrie - Civilistico - 2024"])

                                    31-12-2024  31-12-2023
totale_attivo                       77215809.0  77158352.0
patrimonio_netto                    39001550.0  39296076.0
totale_attivo_circolante            39785638.0  39139925.0
Totale_disponibilità_liquide         7885152.0  10417013.0
Totale_debiti                       36110055.0  36177382.0
Totale_rimanenze                    12434595.0  10231994.0
Totale_crediti_verso_clienti        14796439.0  13346082.0
Totale_debiti_verso_fornitori       12968817.0  11927095.0
Totale_debiti_verso_banche          20429454.0  21238037.0
avviamento                                 NaN         NaN
ricavi_vendite_e_prestazioni        62224926.0  65517637.0
Utile_perdita_esercizio              2695780.0   4381058.0
Risultato_prima_delle_imposte        3481842.0   5593496.0
Totale_ammortamenti_e_svalutazioni   3704196.0   3409007.0
Totale_interessi_e_altri_oneri       1244562.0    587710.0
immobilizzazioni_immateriali         4286506.0   4444433